Download daily financial data set for 50-100 stocks covering a time period of 10 years
For ech day, store the date(yyyymmdd), the adjusted close price (last price of the day) and the volume (num of shares traded)
Can use Python library yfinance

In [3]:
!pip install yfinance

  Using cached yfinance-1.2.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached multitasking-0.0.12.tar.gz (19 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.1-py3-none-any.whl.metadata (8.5 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata (13 kB)
  Using cached protobuf-7.34.0-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)

In [3]:
!pip install lxml

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 80.3 MB/s  0:00:00


In [2]:
import yfinance as yf
from datetime import datetime
import pandas as pd
import requests
from io import StringIO

In [3]:
#Time period to fetch from
start_data = datetime(year=2015, month=1, day=1)


end_data = datetime(year=2026, month = 3, day =9)
#list of stocks
wikipedia_page = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

response = requests.get(wikipedia_page, headers=headers)
response.raise_for_status()

list_of_stocks = pd.read_html(StringIO(response.text))[0]

ticker_symbols = list_of_stocks['Symbol'].tolist()
ticker_symbols_75 = ticker_symbols[:75]


In [4]:
all_data = []
ticker_symbols_75_clean = [t.replace('.', '-') for t in ticker_symbols_75]

data = yf.download(tickers=ticker_symbols_75_clean, start=start_data, end=end_data, group_by='ticker')

#in data: key: [[date, close, volume], [date, close, volume], ...]]
for ticker in ticker_symbols_75_clean:
    hist_data = data[ticker][['Close', 'Volume']].copy()
    hist_data["Ticker"] = ticker
    hist_data.reset_index(inplace=True) #Make date column
    all_data.append(hist_data)
    
    
combined_data = pd.concat(all_data, ignore_index=True)
print(combined_data.head())

[*********************100%***********************]  75 of 75 completed


Price       Date      Close     Volume Ticker
0     2015-01-02  95.679222  2531214.0    MMM
1     2015-01-05  93.521378  4416708.0    MMM
2     2015-01-06  92.524124  4224272.0    MMM
3     2015-01-07  93.194824  3685235.0    MMM
4     2015-01-08  95.428467  3758908.0    MMM


Test set: all data between 20250101 and now
Val set: all data between 20240101 and 20241231
Training data: 2015 to 20231231

In [6]:
train_data = combined_data[combined_data["Date"] <= datetime(year=2023, month=12, day=31)]
validation_data = combined_data[(combined_data["Date"] > datetime(year=2023, month=12, day=31)) & (combined_data["Date"] <= datetime(year=2024, month=12, day=31))]
test_data = combined_data[combined_data["Date"] > datetime(year=2024, month=12, day=31)]

#X_train, y_train = train_data[['Date', 'Ticker', "Volume"]], train_data['Close']
#X_val, y_val = validation_data[['Date', 'Ticker', "Volume"]], validation_data['Close']
#X_test, y_test = test_data[['Date', 'Ticker', "Volume"]], test_data['Close']



Add another column for each time series and in each set
Column; int(1) if closing stock price on the next day is >=3% higher than price on current day, 0 otherwise


In [ ]:

#Takes like 0 sec
train_data["Increase"] = train_data.groupby("Ticker")["Close"].shift(-1) >= 1.03 * train_data['Close'] #Groupby means it looks at the data for each tiocker seperately, the shift means that we look at the data for next step
train_data["Increase"] = train_data["Increase"].astype(int) #Convert boolean to int (True becomes 1, False becomes 0)


""" #takes 4 min
for ticker in ticker_symbols_75:
    data_for_ticker = train_data[train_data['Ticker'] == ticker].reset_index() #Reset index makes hte index into a regular column, so we can refer to the different dates as 0,1,2, etc. instead of like the date
    
    for date_indx in range(len(data_for_ticker)-1):
        closing_today = data_for_ticker.loc[date_indx, 'Close']
        closing_tomorrow = data_for_ticker.loc[date_indx + 1, 'Close']
        
        date_today = data_for_ticker.loc[date_indx, "Date"]
        
        if closing_tomorrow >= 1.03 * closing_today:
            train_data.loc[(train_data["Ticker"] == ticker) & (train_data["Date"] == date_today)]["Increase"] = 1
        else:
            #print(train_data[ticker])
            train_data.loc[(train_data["Ticker"] == ticker) & (train_data["Date"] == date_today)]["Increase"] = 0

"""
print(train_data)


#iloc uses integers, loc uses labels

            

Price        Date       Close     Volume Ticker  Increase
0      2015-01-02   95.679222  2531214.0    MMM         0
1      2015-01-05   93.521378  4416708.0    MMM         0
2      2015-01-06   92.524124  4224272.0    MMM         0
3      2015-01-07   93.194824  3685235.0    MMM         0
4      2015-01-08   95.428467  3758908.0    MMM         0
...           ...         ...        ...    ...       ...
210199 2023-12-22  193.450302   568800.0     BR         0
210200 2023-12-26  195.727127   981100.0     BR         0
210201 2023-12-27  197.907043   537500.0     BR         0
210202 2023-12-28  199.418442   535500.0     BR         0
210203 2023-12-29  199.340927   441100.0     BR         0

[169800 rows x 5 columns]
